In [12]:
# Import all necessary libraries for data analysis and machine learning
import numpy as np  # for numerical operations and arrays
import pandas as pd  # for data manipulation and dataframes
import scipy  # for scientific computing
from scipy import stats  # for statistical tests
from scipy.stats import norm, t, binom, poisson, expon, chi2, f  # specific statistical distributions
import matplotlib.pyplot as plt  # for creating plots and visualizations
import seaborn as sns  # for enhanced statistical visualizations
import statsmodels  # for statistical modeling
import statsmodels.api as sm  # for regression models (OLS, Logit, etc.)
from statsmodels import stats as sm_stats  # for statistical tests
from statsmodels.stats import weightstats as sswa  # for weighted statistics
from statsmodels.formula.api import ols  # for ordinary least squares regression with formulas
from statsmodels.stats.multicomp import pairwise_tukeyhsd  # for post-hoc ANOVA tests
from statsmodels.stats import proportion as ssp  # for proportion tests
from statsmodels.stats.proportion import proportions_ztest  # for z-test on proportions
import os  # for file and directory operations
import warnings  # to manage warning messages
warnings.filterwarnings("ignore")  # suppress warning messages for cleaner output
from scipy.stats import chi2_contingency  # for chi-square test of independence
from sklearn.model_selection import train_test_split  # to split data into training and testing sets
from sklearn.linear_model import LinearRegression  # for linear regression modeling

In [13]:
os.chdir(r'/Users/proxim/Desktop/CDAC-DBDA-coursework/08.advanced-analytics-stats')

In [14]:
# Load new dataset: Algeria Fire data (for predicting fire weather index)
df = pd.read_excel('data/AlgeriaFire.xlsx')

In [15]:
# Display first 5 rows of Algeria Fire dataset to understand the features
df.head()

,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,Classes,Region,FWI
0,29,57,18,0.0,65.7,3.4,7.6,1.3,3.4,0,0,0.5
1,29,61,13,1.3,64.4,4.1,7.6,1.0,3.9,0,0,0.4
2,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0,0,0.1
3,25,89,13,2.5,28.6,1.3,6.9,0.0,1.7,0,0,0.0
4,27,77,16,0.0,64.8,3.0,14.2,1.2,3.9,0,0,0.5


In [16]:
# Separate features (X) and target variable (y)
x = df.drop('FWI', axis=1)  # X = all columns except FWI (Fire Weather Index - what we want to predict)
y = df['FWI']  # y = FWI (our continuous target variable)

# Convert categorical columns to dummy variables
class_dummy = pd.get_dummies(df.Classes, prefix='Classes', drop_first=True).astype(int)  # create dummies for Classes column
reg_dummy = pd.get_dummies(df.Region, prefix='Region', drop_first=True).astype(int)  # create dummies for Region column
x.drop(['Classes', 'Region'], axis=1, inplace=True)  # remove original categorical columns
x = pd.concat([x, class_dummy, reg_dummy], axis=1)  # add dummy variables to feature set

# Split data into training (80%) and testing (20%) sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=20)


# Add constant term for intercept in regression equationprint(mod1.summary())

x_train = sm.add_constant(x_train, prepend=False)# Display regression results including R-squared, coefficients, p-values

# Fit OLS (Ordinary Least Squares) regression model (used for continuous outcomes like FWI)mod1 = sm.OLS(y_train, x_train).fit()

In [17]:
# Import VIF to check for multicollinearity (when predictors are highly correlated with each other)
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [18]:
# Calculate VIF (Variance Inflation Factor) for each feature to detect multicollinearity
vif_data = pd.DataFrame()  # create empty dataframe to store results
vif_data["feature"] = x.columns  # add feature names
vif_data["VIF"] = [variance_inflation_factor(x.values, i) for i in range(x.shape[1])]  # calculate VIF for each feature
# VIF > 10 indicates high multicollinearity (features are too correlated)
print(vif_data)

        feature         VIF
0   Temperature  109.429048
1            RH   32.236045
2            Ws   37.130416
3          Rain    1.655248
4          FFMC  114.994902
5           DMC  205.539005
6            DC   51.257576
7           ISI   11.137955
8           BUI  393.610061
9     Classes_1    7.064774
10     Region_1    3.149722


In [19]:
# Import StandardScaler to normalize numeric features (mean=0, std=1)
from sklearn.preprocessing import StandardScaler

# Select only numeric columns that need scaling
x_nums = x_train[['Rain', 'DC', 'ISI']]
scl1 = StandardScaler()  # create scaler object

# Fit scaler on training data and transform it (learn mean/std, then scale)
x_nums_scaled = scl1.fit_transform(x_nums)

# Transform test data using the same scaling parameters (DO NOT fit on test data to avoid data leakage)x_train = pd.concat([pd.DataFrame(x_nums_scaled, columns=['Rain', 'DC', 'ISI'], index=x_train.index), x_train[['Classes_1', 'Region_1']]], axis=1)

x_test_scaled = scl1.transform(x_test[['Rain', 'DC', 'ISI']])# Combine scaled numeric features with categorical dummy variables


In [20]:
# Create a Linear Regression model using sklearn (simpler than statsmodels)
linreg = LinearRegression()
linreg.fit(x_train, y_train)  # train the model on training data
linreg.coef_  # view coefficients for each feature (how much each feature affects the prediction)
linreg.intercept_  # view intercept (baseline prediction when all features are 0)

np.float64(2.851572901864407)

In [21]:
# Import RidgeCV for Ridge regression with automatic tuning of regularization parameter
from sklearn.linear_model import RidgeCV
ridgecv = RidgeCV(cv=5)  # cv=5 means use 5-fold cross-validation to find best alpha
ridgecv.fit(x_train, y_train)  # train Ridge model (adds penalty to prevent overfitting)
ridgecv.alpha_  # view the best alpha value chosen by cross-validation (controls regularization strength)

np.float64(10.0)

In [22]:
# Import Ridge regression with manual alpha setting
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1)  # alpha=1 controls how much to penalize large coefficients (prevents overfitting)
ridge.fit(x_train, y_train)  # train Ridge model
ridge.coef_  # view coefficients (they'll be smaller than linear regression due to regularization)

array([ 4.98394898e-03, -1.69505551e-02, -6.56028149e-03,  2.49630553e-02,
       -5.77047194e-02, -6.80517317e-04, -1.71145057e-02,  1.12195074e+00,
        3.09600181e-01,  6.37341790e-01, -4.35107818e-01,  0.00000000e+00])